# Video Game Commercial Success Prediction & Market Analysis

The goal of this project is to deliver a clear commercial narrative that game studios and publishers can use to minimize financial risk before greenlighting a new game's production.

When looking into creating new games studios and publishers investigate what has been historically successful. By analyzing characteristics of games that have already been released, we can help guide new game ideas to the right publishers to help propagate the game to a better sales pattern, allowing both the publisher and game developer to maximize their investment.

#### Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sqlalchemy import create_engine, text
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

## Step 1: Environment Setup & In-Memory Database Initialization (SQL)

To start, we will create an in-memory database using SQLAlchemy. 
By doing this, the data pipeline can be run anywhere.

In [ ]:
# Create an in memory SQLite database
engine = create_engine('sqlite:///:memory:')

In [ ]:
# Load data into pandas dataframe
df_raw = pd.read_csv('datasets/Video_Games_Sales.csv')
df_raw.info()

In [ ]:
# ingest raw data into the in-memory SQL database
df_raw.to_sql('raw_game_sales', con=engine, index=False, if_exists='replace')

In [ ]:
# Run SQL query directly against the database
query = """
SELECT count(*)
FROM raw_game_sales
"""

In [ ]:
with engine.connect() as conn:
    results = conn.execute(text(query))
    print(f'Successfully ingested {results.scalar()} records into in-memory table "raw_game_sales".')

## Step 2: Cleaning and Transformation (SQL)

Next we will pull the columns we need for the analysis. We will also filter out the `NULL` values from the `critic_scores`, `global_sales`, `year_of_release`, `user_score`, and `publisher`.

In [ ]:
# Query to pull only the needed columns from database
query = """
SELECT platform as platform,
    year_of_release as release_year,
    genre as genre, 
    publisher as publisher,
    global_sales as global_sales
FROM raw_game_sales
WHERE global_sales IS NOT NULL
AND year_of_release IS NOT NULL
AND publisher IS NOT NULL;
"""

In [ ]:
with engine.connect() as conn:
    results = conn.execute(text(query))
    df_cleaned = pd.DataFrame(results)

In [ ]:
df_cleaned.info()

In [ ]:
# Remove rows containing NaN values
df_cleaned = df_cleaned.dropna()

In [ ]:
# Confirm change
df_cleaned.info()

In [ ]:
# convert release year to int type
df_cleaned['release_year'] = df_cleaned['release_year'].astype(int)

## Step 3: Exploratory Data Analysis & Feature Engineering (Python)

Now we need to set up the binary classification target: `Is_Hit` = `1` if `global_sales` $\geq 1.0$ else `0`.

In [ ]:
# 80th percentile
overall_80th = df_cleaned['global_sales'].quantile(.80)
print(f'Overall 80th percentile: {overall_80th}')

In [ ]:
# groub by 'release_year' and  80th percentile threshold within each year
df_cleaned['release_']

Next we'll calculate the baseline class balance ratio (percetage of hits vs. non-hits).

In [ ]:
# Calulate the average and multiply by 100 to get the percentage
counts = df_cleaned['is_hit'].value_counts(normalize=True) * 100

print('Balance Ratios:')
print(f'hit:      {counts.get(1.0) :.2f}%')
print(f'not_hit: {counts.get(0.0) : .2f}%')

Above, we calculate the class balance ratios using `normalize=True`, then multiply by 100 to yield relative percentages rather than raw counts. By referencing values with explicit class labels (`1` for hit, `0` for non-hit) rather than positional indexing, the calculation remains accurate and robust to unexpected shifts in class distribution in future data ingestions.

#### Feature Selection & Preprocessing

In [ ]:
# column name reference
df_cleaned.columns

In [ ]:
# target vector
y = df_cleaned['is_hit']

# predictive features
features = ['platform','release_year','genre','publisher']

X = df_cleaned[features]

> Note: The regional sales columns (`na_sales`, `eu_sales`, `jp_sales`, `other_sales`) and `global_sales` are retained in the cleaned SQL dataset for exploratory data analysis and regional marketing profiling. To prevent target leakage, all sales metrics will be strictly excluded from the feature matrix (`X`) prior to model training.

Next we will handle publishers with high-cardinality. There can be publishers with only 1 to 2 historical titles, which can introduce hundreds of dummy columns when encoded. Grouping these publishers into an `other` category will keep our feature matrix manageable.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

In [ ]:
# train/test split on raw data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
# keep top 20 publishers, group the rest as `other`
top_publishers = X['publisher'].value_counts().nlargest(20).index

X_train['publisher'] = X_train['publisher'].apply(lambda x: x if x in top_publishers else 'other')
X_test['publisher'] = X_test['publisher'].apply(lambda x: x if x in top_publishers else 'other')
X.sample(10)

In [ ]:
# define column groups
num_cols = ['release_year']
cat_cols = ['platform', 'genre', 'publisher']

In [ ]:
# build preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), cat_cols)
    ]
)

In [ ]:
# fit on X_train then transform both splits
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

#### Exploratory Analysis

Since we've calculated the baseline target class balance(`is_hit` ratio of $81/19$). We will generate some visual plots to verify the features belong in our model.

#### Correlation Matrix & Feature Distribution Plots

In [ ]:
# numeric correlation matrix
plt.figure(figsize=(10,6))
numeric_cols = ['na_sales','eu_sales','jp_sales','other_sales','global_sales']
sns.heatmap(df_cleaned[numeric_cols].corr(), annot=True, cmap='Blues',fmt='.2f')
plt.title('Feature Correlation Matrix')
plt.show()

# score distribution plot
#fig, axes = plt.subplots(1,2, figsize=(12,5))
#sns.histplot(df_cleaned['critic_score'], kde=True, ax=axes[0], color='skyblue').set(title='Critic Score Distribution')
#sns.histplot(df_cleaned['user_score'], kde=True, ax=axes[1], color='salmon').set(title='User Score Distribution')
#plt.tight_layout()
#plt.show()

####  Regional Sales Summary Metrics

In [ ]:
# regional sales summary statistics
regional_cols = ['na_sales','eu_sales','jp_sales','other_sales','global_sales']
regional_summary = df_cleaned[regional_cols].describe().T[['mean','std','min','50%','max']]
regional_summary.columns = ['Mean ($M)', 'Std Dev', 'Min', 'Median', 'Max']

display(regional_summary)

Across all regions, the sales data exhibits high right-skewness. The `Median` regional sales are significantly lower than the `mean` sales. This confirms that a small percentage of blockbuster titles accounts for the vast majority of commercial volume.

North American sales (`na_sales`) demonstrate the strongest individual regional correlation with global success (`global_sales`), averaging $\$0.39M$ per title. European sales (`eu_sales`) follow closely as the second largest contributor.

Japanese sales (`jp_sales`) show a significantly lower correlation with Western sales performance. This highlights distinct regional consumer preferences: titles that perform exceptionally well in Western markets do not automatically replicate that success in Japan, and vice versa.

`critic score` demonstrates a stronger positive correlation with `global_sales` than `user_score`. This indicates that professional critic evaluations serve as a stronger leading indicator of commercial viability during a title's initial launch window than consumer review scores.

### Step 4: Predictive Modeling & Hypothesis Testing (Python)

For the model, I've chosen to use `LogisticRegression` to predict commercial hits.

The target metric (`is_hit`) is binary-categorized as `1` for games with global sales $\ge \$1.0\text{M}$ and `0`. Logistic Regression uses the sigmoid function to map feature inputs (e.g., `scores`, `genre`, `platform`, `publisher`) directly into probabilities between `0` and `1`, which helps avoid the out-of-bounds prediction errors inherent to linear regression models.

Unlike 'Black box'  machine learning models, Logistic regression provides clear, quantifiable feature coefficients ($\beta$). Converting these coefficients to **Odd Ratios** ($e^{\beta}$) allows stakeholders to measure exactly how individual factors, such as a 10-point bump in review scores or release on a specific console, increase or decrease a game's likelihood of becoming a commercial success.

In [ ]:
# create model
model = LogisticRegression(class_weight='balanced', max_iter=10000, random_state=42)

In [ ]:
# fit model
model_fit = model.fit(X_train_processed, y_train)

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix, classification_report

In [ ]:
from sklearn.metrics import precision_recall_curve

In [ ]:
# generate predictions based on custome threshold
y_prob = model.predict_proba(X_test_processed)[:, 1]

custom_threshold = .60

y_pred_custom = (y_prob >= custom_threshold).astype(int)

In [ ]:
# calculate the precision recall curve
precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob)

In [ ]:
# plot to see recall and precision
plt.figure(figsize=(8,5))
plt.plot(thresholds, precisions[:-1], "b--", label='Precision')
plt.plot(thresholds, recalls[:-1], "g--", label='recall')
plt.xlabel('Decision Threshold')
plt.ylabel('Score')
plt.title('Precision and Recall vs. Decision Threshold')
plt.legend(loc='center left')
plt.grid(True)
plt.show()

In [ ]:
# compute metrics
acc = accuracy_score(y_test, y_pred_custom) * 100
prec = precision_score(y_test, y_pred_custom) * 100
rec = recall_score(y_test, y_pred_custom) * 100
f1 = f1_score(y_test, y_pred_custom) * 100

In [ ]:
print('--- Model Performance Metrics ---')
print(f'Accuracy: {acc:.4f}')
print(f'Precision: {prec:.4f}')
print(f'Recall: {rec:.4f}')
print(f'F1-Score: {f1:.4f}')

In [ ]:
# print formatted classification report
print('--- Classification Report ---')
print(classification_report(y_test, y_pred_custom, target_names=['Not Hit', 'Hit']))

In [ ]:
# plot confusion matrix
cm = confusion_matrix(y_test, y_pred_custom)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Not Hit', 'Hit'],
           yticklabels=['Not Hit', 'Hit'])
plt.title('Final Model Confusion Matrix')
plt.xlabel('Prediction Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.show()